In [85]:
import pandas as pd
import numpy as np
import seaborn as sns
import datetime
import os




In [92]:
rawlegaldesc = pd.read_csv(r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\deed transfer test data\Deed transfer legal desc/20250811_RPI_LegalDesc.txt",sep="|")

rawdeedtransferdate = pd.read_csv(r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\deed transfer test data\DeedTransfer date/20250811_RPI_Instruments.txt",sep="|")

rawdeedtransfernames = pd.read_csv(r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\deed transfer test data\DeedTransfer name/20250811_RPI_Names.txt",sep="|")

#check column names
print(rawlegaldesc.columns)
print(rawdeedtransferdate.columns)
print(rawdeedtransfernames.columns)

#look for duplicated values
print("Legal Desc duplicates:", rawlegaldesc["File No"].duplicated().sum())
print("Date duplicates:", rawdeedtransferdate["File No"].duplicated().sum())
print("Names duplicates:", rawdeedtransfernames["File No"].duplicated().sum())


import pandas as pd

# Clean column names + key
for df in [rawlegaldesc, rawdeedtransferdate, rawdeedtransfernames]:
    df.columns = df.columns.str.strip()
    df["File No"] = df["File No"].astype(str).str.strip()

# 1) FILTER to sellers only
seller_rows = rawdeedtransfernames[
    rawdeedtransfernames["NType"].astype(str).str.strip() == "1"
].copy()

# 2) AGGREGATE seller names to one row per File No
seller_names = (seller_rows
    .assign(Name=lambda d: d["Name"].astype(str).str.strip())
    .groupby("File No", as_index=False)["Name"]
    .agg(lambda s: "; ".join(sorted(set(s.dropna()))))
    .rename(columns={"Name": "Seller_Name"})
)

# 3) Dedup legal desc (you had 1 duplicate File No)
legal_1 = rawlegaldesc.drop_duplicates(subset=["File No"], keep="first")

# 4) Merge in date + seller names
merged = (legal_1
    .merge(rawdeedtransferdate, on="File No", how="left")
    .merge(seller_names, on="File No", how="left")
)

# 5) Make the output column exactly what you want
# (prevents confusion if other "Name" columns exist)
merged["Name"] = merged["Seller_Name"]
merged = merged.drop(columns=["Seller_Name"], errors="ignore")

# 6) Quick proof for your specific example file
print(seller_rows[seller_rows["File No"] == "RP-2025-311874"][["File No","Name","NType"]])
print(merged.loc[merged["File No"] == "RP-2025-311874", ["File No","Name"]])

# 7) Export
out_path = r"C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\deed transfer test data\Merged data ready for import\final_merged_sellers_only.csv"
merged.to_csv(out_path, index=False, encoding="utf-8-sig")
print("Saved:", out_path)


Index(['File No', 'SubDivision', 'Section', 'Lot', 'Block', 'Misc', 'Map Page',
       'Unit', 'Abstract', 'OutLot', 'Tract', 'Reserve', 'Comment'],
      dtype='object')
Index(['File No', 'Document Type', 'FileDate', 'Film Code No.',
       'No. of Pages'],
      dtype='object')
Index(['File No', 'Name', 'NType', 'Document Type'], dtype='object')
Legal Desc duplicates: 1
Date duplicates: 0
Names duplicates: 4948
          File No          Name  NType
6  RP-2025-311874  HUGHES PETER      1
          File No          Name
2  RP-2025-311874  HUGHES PETER
Saved: C:\Users\howar\OneDrive\Documents\BU Mod2\699 pt2\Real Estate\deed transfer test data\Merged data ready for import\final_merged_sellers_only.csv
